# GEIGER-1911 · 2 — Which angles

Notebook 1 left a factor of a hundred million between the two atoms at five degrees. It
looks like the question is already answered and all that remains is to point a detector
somewhere and look.

It is not, and the reason is the whole of experimental design. A factor of a hundred
million at five degrees is a statement *made inside a model* — it assumes the width of
the multiple-scattering core, which is the very thing a defender of the diffuse atom
would revise. This notebook asks four questions in the order they constrain each other,
and the fourth is the one that decides where the detector goes.

| | decides | reaches into |
|---|---|---|
| **1** | what can this design estimate at all? | `design.sensitivity_matrix`, `design.estimable_combinations` |
| **2** | how precisely, and what can it never pin down? | `design.fisher_information`, `design.expected_posterior_sd`, `design.identifiability_ridge` |
| **3** | where is the information, if the model is right? | `weight_of_evidence`, `design.eig_gaussian` |
| **4** | where is the information that survives being wrong? | `scattering.surviving_evidence` |

In [ ]:
import sys

sys.path.insert(0, ".")

import math

import numpy as np
import plotly.graph_objects as go

import scattering as S
from axiom.core import Unsupported
from axiom.design import (
    EstimabilityReport, FisherInformation, IdentifiabilityRidge, IdentifyingDesign,
    design_to_identify, eig_gaussian, estimable_combinations, expected_posterior_sd,
    fisher_information, identifiability_ridge, observation_from_model, sensitivity_matrix,
)
from axiom.surface import Design

from axiom.display import enable, table

enable();  # every axiom result renders itself from here on

model, surface = S.model(), S.surface()
PARAMS = ("log_a", "lam", "log_c", "log_w", "log_b")

# Every claim below is local to a parameter point (information on a nonlinear surface
# always is). VISIBLE is a charge ball big enough for this beam to see the edge of --
# the only place where "how precisely is R measured" is a question with an answer.
VISIBLE = S.truth(math.log(0.30))
print(f"the point the design math is done at: a charge ball of "
      f"{S.radius_of(VISIBLE['lam']) * 1e15:.0f} fm, whose scattering dies at "
      f"{math.degrees(2 * math.asin(math.exp(VISIBLE['lam']))):.0f} degrees")

## 1 · What can each design estimate at all

Before precision, rank. `sensitivity_matrix` differentiates every prediction with respect
to every parameter and `estimable_combinations` reads off what the design can and cannot
separate. The parameters are already logarithms, so `scaling="absolute"` here gives
*linear* combinations of logs — which are ratios and products of the physical quantities.

Four candidate campaigns, each a caricature of a real one.

In [ ]:
CAMPAIGNS = {
    "small angles only (0.5-2 deg)": S.station([0.5, 1.0, 2.0], 1e-8, 3600.0),
    "a mid bank only (5-45 deg)": S.station([5.0, 10.0, 20.0, 45.0], 1e-5, 3600.0),
    "wide angles only (60-150 deg)": S.station([60.0, 90.0, 150.0], S.OMEGA_MAX, 3600.0),
    "a sweep, foil never removed": S.station([0.5, 2.0, 5.0, 20.0, 60.0, 150.0], 1e-5, 3600.0),
}

rows = []
for label, data in CAMPAIGNS.items():
    obs = observation_from_model(model, data, noise_sd=1.0, label=label)
    matrix = sensitivity_matrix([obs], VISIBLE, parameters=PARAMS, scaling="absolute")
    report = estimable_combinations([obs], VISIBLE, parameters=PARAMS, scaling="absolute")
    assert not isinstance(matrix, Unsupported) and isinstance(report, EstimabilityReport)
    flat = [c.label for c in report.symmetries] or ["-"]
    rows.append([label, f"{report.rank}/5", ", ".join(matrix.zero_columns) or "-", ", ".join(flat)])
table(rows, headers=("campaign", "rank", "dead columns", "flat directions"))

Three of the four campaigns are rank-deficient, and each fails differently.

**Small angles only** cannot see the background at all — `log_b` is a dead column, meaning
the mean does not move with it, because at half a degree the counter is being hit ten
billion times harder by the beam than by anything stray. A campaign that never leaves the
core therefore cannot say what its own background is, and a wide-angle count it has not
taken is exactly what a background would be confused with. This is the 1909 measurements
in one line.

**A mid bank only** cannot see the core: `log_c` and `log_w` are both flat. It measures
the tail beautifully and has nothing to say about the thing the tail has to be
distinguished from.

**Wide angles only** is rank-deficient in the other direction.

Only the sweep is full rank — and it still has not measured the background, because the
foil never came out. That is not a rank failure; it is a precision failure, which is
section 2.

In [ ]:
with_foil_out = S.station([0.5, 2.0, 5.0, 20.0, 60.0, 150.0, 90.0],
                          [1e-8] * 3 + [1e-5] * 3 + [S.OMEGA_MAX],
                          3600.0, [1.0] * 6 + [0.0])
report = estimable_combinations([observation_from_model(model, with_foil_out, noise_sd=1.0)],
                                VISIBLE, parameters=PARAMS, scaling="absolute")
assert isinstance(report, EstimabilityReport)
print(f"the sweep plus one foil-out run: rank {report.rank}/5")
print("best-determined combinations, and how sharply the design pins each:")
table(
    [[c.label, f"{c.score:.3f}"] for c in report.estimable[:3]],
    headers=("combination", "score"),
)

## 2 · How precisely — and the one thing it can never pin down

Rank says a parameter is reachable. `fisher_information` says how far. Both are local, so
the honest thing is to ask at both hypotheses and at a point between them.

In [ ]:
sweep = S.station([1.0, 2.5, 5.0, 10.0, 20.0, 45.0, 90.0, 150.0, 90.0],
                  list(S.widest_aperture([1.0, 2.5, 5.0, 10.0, 20.0, 45.0, 90.0, 150.0],
                                         S.HARD_CENTRE)) + [S.OMEGA_MAX],
                  20 * 3600.0, [1.0] * 8 + [0.0])

rows = []
for label, point in (("a visible charge ball", VISIBLE),
                     ("the hard centre", S.HARD_CENTRE),
                     ("the diffuse atom", S.DIFFUSE)):
    info = fisher_information(surface, sweep, point, 1.0, method="finite")
    assert isinstance(info, FisherInformation)
    sds = expected_posterior_sd(S.PRIOR_SDS, info)
    assert isinstance(sds, dict)
    about_lam = info.as_array()[info.index("lam"), info.index("lam")]
    rows.append(
        [label, f"{about_lam:.4g}", str(info.detail.get("round_off_columns", "-")),
         "  ".join(f"{k} {v:.3g}" for k, v in sds.items())]
    )
table(rows, headers=("world", "information about lam", "dead columns", "expected posterior sds"))

Read the `lam` column. At a visible charge ball the design carries a few hundred thousand
units of information about it and measures it to a fifth of a per cent. **At the hard
centre the information is twelve orders of magnitude smaller and `lam` comes back at its
prior standard deviation of 6.0, untouched.**

That is not a numerical complaint. It is notebook 1's ceiling arriving in the design
math, before any apparatus is built: a charge the beam never reaches is a charge whose
size does not enter any prediction, so **if the hard centre is right, this experiment
cannot measure the nuclear radius and will not pretend to.** What it returns is a bound.

The diffuse atom is deader still. There `log_a` and `lam` are both named as *dead
columns* — if the tail is cut off at a fortieth of a degree, its amplitude is not
measurable either. The experiment can refute the diffuse atom; it can never *fit* it.

In [ ]:
anchored = S.station([0.5, 1.0, 2.0, 90.0], [1e-8] * 3 + [S.OMEGA_MAX],
                     20 * 3600.0, [1.0, 1.0, 1.0, 0.0])
ridge = identifiability_ridge(surface, anchored, VISIBLE, 1.0, method="finite")
assert isinstance(ridge, IdentifiabilityRidge)
print(f"small angles, background measured: condition number {ridge.condition_number:,.3g}")
print("  flattest direction:", {k: round(v, 3) for k, v in ridge.direction.items()})

paired = identifiability_ridge(surface, anchored, VISIBLE, 1.0,
                               pairs=(("log_a", "log_c"),), method="finite")
print("  asking for a pairwise correlation ->", type(paired).__name__)
if isinstance(paired, Unsupported):
    print("   ", paired.reason[:120])

ridge = identifiability_ridge(surface, sweep, VISIBLE, 1.0,
                              pairs=(("log_a", "log_c"), ("lam", "log_a")), method="finite")
assert isinstance(ridge, IdentifiabilityRidge)
print(f"\nthe full sweep: condition number {ridge.condition_number:,.0f}")
print("  flattest direction:", {k: round(v, 3) for k, v in ridge.direction.items()})
print("  carried mostly by:", ridge.ridge_parameters)
table(
    [[pair[0], pair[1], f"{r:+.3f}"] for pair, r in zip(ridge.pairs, ridge.correlations)],
    headers=("parameter", "against", "posterior correlation"),
)

The small-angle campaign's flattest direction is worth reading slowly, because it is the
opposition's entire case written as an eigenvector:

> raise the tail's amplitude, shrink the charge, lower the core and widen it — all at
> once, in these proportions — and **nothing any small-angle detector sees will change.**

That is a four-way degeneracy with a condition number near $10^{16}$, which is a singular
matrix in every practical sense: `identifiability_ridge` will not report a pairwise
correlation off it, because a singular matrix has no covariance to take one from. Adding
the foil-out run fixes the background and does not touch the degeneracy at all. Section 4
is this eigenvector let loose.

The full sweep's flattest direction, by contrast, is carried by the core's width and
amplitude — the two things nobody is asking about — while `lam` contributes almost
nothing to it. A design whose weakest direction is a nuisance is a design in good shape.

## 3 · Where the information is, if the model is right

Now the quantity the design is actually chosen on. For a choice between *two* hypotheses
the expected information gain has a closed form: the Kullback-Leibler divergence between
two Poisson counts,

$$\text{nats} = \mu_H \log\frac{\mu_H}{\mu_D} - (\mu_H - \mu_D)$$

which is the expected log Bayes factor in favour of the hard centre when the hard centre
is true, in the same nats `design.eig_gaussian` reports in.

Two constraints decide the aperture before the statistics do, and they bind at opposite
ends of the range. A human at a scintillation screen counts about ninety flashes a minute
before starting to miss them, so at small angles the aperture must be a pinhole. Past
about eighty degrees the apparatus runs out of screen. In between, every station is
counting at the same capped rate, and what differs is the *worth of each count*.

In [ ]:
grid = np.geomspace(0.4, 175.0, 220)
aperture = S.widest_aperture(grid, S.HARD_CENTRE)
mu_h = S.rate_per_steradian(grid, S.HARD_CENTRE) * aperture * 3600.0
mu_d = S.rate_per_steradian(grid, S.DIFFUSE) * aperture * 3600.0
nats = S.weight_of_evidence(mu_h, mu_d)

fig = S.figure("What an hour at each angle is worth", "scattering angle",
               "expected log Bayes factor (nats / hour)", height=440)
fig.add_trace(go.Scatter(x=grid, y=nats, name="if the model of the core is right",
                         line={"color": S.HARD_COLOR, "width": 3}))
fig.update_yaxes(type="log", exponentformat="power")
S.degrees_axis(fig, log=True)
fig.add_vrect(x0=0.4, x1=3.0, fillcolor=S.TRUTH_COLOR, opacity=0.07, line_width=0,
              annotation_text="both atoms<br>agree here", annotation_position="top left")
fig.add_vrect(x0=80.0, x1=175.0, fillcolor=S.TRUTH_COLOR, opacity=0.07, line_width=0,
              annotation_text="the screen<br>runs out", annotation_position="top right")
fig.show()

peak = grid[int(np.argmax(nats))]
print(f"peak at {peak:.1f} degrees, {nats.max():,.0f} nats/hour")
print(f"an hour at 150 degrees is worth {np.interp(150.0, grid, nats):,.0f} nats, "
      f"{nats.max() / np.interp(150.0, grid, nats):.0f}x less")

On this reading the answer is unambiguous, and it is not the one folklore gives: **the
information is at five degrees**, an hour there is worth twenty times an hour at a hundred
and fifty, and the back-angle station looks like a waste of a very scarce two hundred
hours.

It is not the counts that differ. Both stations are rate-capped, so both collect the same
5,400 flashes an hour; what differs is what the diffuse atom has to say about them — a
millionth of a count at five degrees, sixteen counts at a hundred and fifty.

In [ ]:
# The same number through design.eig_gaussian, which is what the rest of axiom uses:
# a design that pins lam to sd `se` against a prior sd of 6.0 gains this many nats.
station_sd = {}
for angle in (5.0, 20.0, 90.0, 150.0):
    one = S.station([angle], S.widest_aperture([angle], S.HARD_CENTRE), 20 * 3600.0)
    info = fisher_information(surface, one, VISIBLE, 1.0, method="finite")
    assert isinstance(info, FisherInformation)
    i = info.index("lam")
    station_sd[angle] = float(np.sqrt(1.0 / info.as_array()[i, i]))

table(
    [
        [f"{angle:.0f}", f"{se:.4g}", f"{eig_gaussian(S.PRIOR_SDS['lam'], se):.2f}"]
        for angle, se in station_sd.items()
    ],
    headers=("angle (deg)", "sd of lam, 20 h alone", "nats about lam"),
)
print("\nMeasuring the radius and refuting the diffuse atom are different jobs, and they")
print("do not rank the angles the same way.")

## 4 · Where the information is when the model is wrong

Everything in section 3 is conditional on the width of the multiple-scattering core. That
width is a *fitted* quantity, and it is exactly what a defender of the diffuse atom would
attack: **your core is wider than you think.**

The defence is not free. The beam is conserved, so a core $f$ times wider is $f^2$ times
lower at the peak — an adversary who widens the core to explain a wide-angle count makes
the small-angle counts come out wrong. So there is a worst case rather than a slippery
slope, and it can be found by minimizing.

`surviving_evidence` does that: for each station, the least evidence it still carries
after the diffuse atom is allowed to pick the most convenient core it can, up to thirty
times the fitted width. Thirty times is twenty degrees rms — five times wider than any
plate measurement of the period, and a generous gift to the opposition.

In [ ]:
survives, worst = S.surviving_evidence(grid, aperture, 3600.0)

fig = S.figure("The same hour, after the opposition has chosen its own core",
               "scattering angle", "expected log Bayes factor (nats / hour)", height=460)
fig.add_trace(go.Scatter(x=grid, y=nats, name="if the model of the core is right",
                         line={"color": S.HARD_COLOR, "width": 3}))
fig.add_trace(go.Scatter(x=grid, y=survives, name="the worst core the opposition can pick",
                         line={"color": S.ACCENT, "width": 3}))
fig.update_yaxes(type="log", exponentformat="power")
S.degrees_axis(fig, log=True)
fig.show()

The two curves are a different experiment.

Where the information was, it evaporates: from two to forty-five degrees the opposition
takes away better than 99 per cent of it, because a core wide enough to reach those angles
is a core nobody has yet ruled out. Only at the far right do the curves rejoin, and **at a
hundred and fifty degrees the evidence does not move at all** — not by a part in ten
thousand — because a core wide enough to reach a hundred and fifty degrees has had its
peak destroyed by the $f^2$ penalty and is refuted somewhere else instead.

In [ ]:
angles = np.array([1.0, 2.5, 5.0, 10.0, 20.0, 45.0, 90.0, 150.0])
ap = S.widest_aperture(angles, S.HARD_CENTRE)
full = S.weight_of_evidence(S.rate_per_steradian(angles, S.HARD_CENTRE) * ap * 3600.0,
                            S.rate_per_steradian(angles, S.DIFFUSE) * ap * 3600.0)
kept, which = S.surviving_evidence(angles, ap, 3600.0)

table(
    [
        [f"{a:.1f}", f"{f:,.0f}", f"{k:,.0f}", f"{k / f:.1%}", f"{w:.1f}x"]
        for a, f, k, w in zip(angles, full, kept, which)
    ],
    headers=("angle (deg)", "nats/h if right", "nats/h if argued with", "kept", "worst core"),
)

### How far back the witness has to sit

The thirty-fold allowance was a choice, and the answer moves with it. Somebody who will
concede only a threefold error in the core needs a much less awkward apparatus than
somebody arguing with a sceptic who will concede thirty. That trade is a curve, and it is
the design's actual specification for the wide-angle station: **sit past the angle no
admissible core can reach.**

In [ ]:
scan = np.geomspace(2.0, 179.0, 300)
scan_ap = S.widest_aperture(scan, S.HARD_CENTRE)
scan_full = S.weight_of_evidence(S.rate_per_steradian(scan, S.HARD_CENTRE) * scan_ap * 3600.0,
                                 S.rate_per_steradian(scan, S.DIFFUSE) * scan_ap * 3600.0)

rows = []
for cap in (3.0, 5.0, 10.0, 20.0, 30.0, 50.0):
    keeps, _ = S.surviving_evidence(scan, scan_ap, 3600.0,
                                    factors=np.geomspace(0.3, cap, 60))
    safe = keeps / scan_full > 0.99
    breached = np.where(~safe)[0]
    beyond = ("nowhere in range" if breached.size and breached[-1] + 1 >= scan.size
              else f"{scan[breached[-1] + 1]:.0f} deg" if breached.size else f"{scan[0]:.0f} deg")
    rows.append([f"{cap:.0f}x", f"{math.degrees(S.CORE_WIDTH * cap):.1f}", beyond])
table(rows, headers=("the opposition may claim a core", "rms (deg)", "immune beyond"))
print("\nTo be immune against a core thirty times the fitted one, the witness has to sit")
print("past 130 degrees. Which is, to within the width of a slit, where Geiger and")
print("Marsden put theirs.")

So the design has to buy two different things, and no single angle sells both.

* Five to forty-five degrees is where the **information** is. It is also where the
  information is contingent on a model of the core, and a campaign that lives there is
  arguing about a fitted curve.
* Sixty to a hundred and fifty degrees is where the **evidence that cannot be argued with**
  is. It is fifty times slower and it is the only part of the experiment whose conclusion
  does not depend on being right about something else.
* Half a degree to three degrees buys neither. It buys the *right to run the argument at
  all*: it is the only place the core's width and the tail's amplitude can be measured, and
  without them the adversary in section 4 is unconstrained rather than merely generous.
* A **foil-out run** buys the background, which nothing else measures and on which every
  wide-angle claim rests.

Four roles. That is the design, and it is worth checking against what the optimizer says
when it is asked for only one of them.

In [ ]:
candidates = [(float(np.radians(a)), float(o), 20 * 3600.0, 1.0)
              for a, o in zip(angles, S.widest_aperture(angles, S.HARD_CENTRE))]
candidates.append((float(np.radians(90.0)), S.OMEGA_MAX, 20 * 3600.0, 0.0))
grid_design = Design(treatments=("theta", "omega", "exposure", "foil"),
                     points=tuple(candidates), kind="angle_grid")

best = design_to_identify(surface, grid_design, VISIBLE, 1.0, target="lam", n=10,
                          prior_sds=S.PRIOR_SDS, seed=0, method="finite")
assert isinstance(best, IdentifyingDesign)
chosen = ["foil out" if p[3] == 0.0 else f"{math.degrees(p[0]):.0f}d" for p in best.design.points]
print("design_to_identify, asked only to pin lam, spends all ten slots on:")
print("  ", ", ".join(sorted(set(chosen), key=chosen.index)))
print(f"   expected sd of lam {best.expected_sd:.4f} against a prior of {S.PRIOR_SDS['lam']}")
print("   and everything else:",
      {k: round(v, 3) for k, v in best.expected_sds.items() if k != "lam"})

Asked to pin one parameter, the point-exchange optimizer buys two angles and nothing
else — and hands back a design in which the core, the core's width and the background all
come back at their prior standard deviations, untouched. It is the sharpest possible
measurement of a number it cannot defend.

This is the same trap notebook 2 of the tutoring case study walks into from the other
side: *a design chosen to be optimal for the model you hope is right cannot tell you
whether it is.* The plan below pays about a factor of two in the precision of `lam` and
buys, with it, every quantity the argument will actually turn on.

In [ ]:
rows = []
for st in S.PLAN:
    ap_here = float(S.widest_aperture([st.theta_deg], S.HARD_CENTRE)[0])
    rows.append([f"{st.theta_deg:.1f}", "in" if st.foil else "out", st.role,
                 f"{st.hours:.0f}", f"{ap_here:.3g}"])
rows.append(["", "", "total", f"{sum(s.hours for s in S.PLAN):.0f}", ""])
table(rows, headers=("angle (deg)", "foil", "role", "hours", "aperture (sr)"))

plan = S.plan_data()
info = fisher_information(surface, plan, VISIBLE, 1.0, method="finite")
assert isinstance(info, FisherInformation)
sds = expected_posterior_sd(S.PRIOR_SDS, info)
assert isinstance(sds, dict)
print("\nwhat the plan expects to know afterwards, as posterior sds in log units:")
table(
    [[name, f"{sd:.4f}", S.PRIOR_SDS[name]] for name, sd in sds.items()],
    headers=("parameter", "posterior sd", "prior sd"),
)

foil_in = [s for s in S.PLAN if s.foil == 1.0]
a_in = np.array([s.theta_deg for s in foil_in])
o_in = S.widest_aperture(a_in, S.HARD_CENTRE)
t_in = np.array([s.seconds for s in foil_in])
kept_plan, _ = S.surviving_evidence(a_in, o_in, t_in)
total = S.weight_of_evidence(S.rate_per_steradian(a_in, S.HARD_CENTRE) * o_in * t_in,
                             S.rate_per_steradian(a_in, S.DIFFUSE) * o_in * t_in)
print(f"\nexpected evidence, model believed:  {total.sum():,.0f} nats "
      f"(log10 Bayes factor {total.sum() / math.log(10):,.0f})")
print(f"expected evidence, model attacked:  {kept_plan.sum():,.0f} nats "
      f"(log10 Bayes factor {kept_plan.sum() / math.log(10):,.0f})")
print(f"of which the two wide stations carry {kept_plan[-2:].sum() / kept_plan.sum():.0%}")

## What notebook 2 established

* **A small-angle campaign cannot measure its own background**, and cannot separate the
  amplitude of the tail from the amplitude of the core. Both come out of
  `estimable_combinations` and `identifiability_ridge` as rank statements, before any
  data.
* **At the hard centre, `lam` is a dead column.** The design math names the thing notebook
  1 derived from geometry: this experiment returns an upper bound on the size of the
  positive charge, never a value.
* If the model of the core is right, **the information is at five degrees** and wide angles
  are twenty times slower.
* If it is not, five degrees loses better than 99.9 per cent of its evidence and **a
  hundred and fifty degrees loses none**. After the opposition has had its say, the two
  wide-angle stations carry two thirds of what is left, and the angle the witness has to
  sit past is a computable function of how wrong the core is allowed to be.

The plan is four kinds of station: an **anchor** at one to three degrees, a **bank** from
five to forty-five, a **witness** at ninety and a hundred and fifty, and a **foil-out**
run. Notebook 3 works out what shape to cut the hole.